In [2]:
import sqlite3
from pathlib import Path

# Create or open a local SQLite database file.
db_path = Path("sqlite_practice.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Start fresh so the cell is safe to run multiple times.
cursor.execute("DROP TABLE IF EXISTS enrollments")
cursor.execute("DROP TABLE IF EXISTS students")
cursor.execute("DROP TABLE IF EXISTS courses")

# Create 3 related tables.
cursor.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    age INTEGER,
    city TEXT
)
""")

cursor.execute("""
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    fee INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrolled_on TEXT DEFAULT CURRENT_DATE,
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
)
""")

# Insert sample records.
students = [
    (1, "Aarav", 20, "Pune"),
    (2, "Meera", 21, "Mumbai"),
    (3, "Kabir", 22, "Delhi"),
    (4, "Anaya", 20, "Pune"),
]

courses = [
    (101, "Python", 5000),
    (102, "SQL", 3500),
    (103, "Data Science", 8000),
]

enrollments = [
    (1, 1, 101),
    (2, 2, 102),
    (3, 3, 103),
    (4, 4, 101),
    (5, 2, 101),
]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", students)
cursor.executemany("INSERT INTO courses VALUES (?, ?, ?)", courses)
cursor.executemany("INSERT INTO enrollments (enrollment_id, student_id, course_id) VALUES (?, ?, ?)", enrollments)
conn.commit()

# Helper to display query results clearly.
def show_query(title, query, params=()):
    print(f"\n{title}")
    print("-" * len(title))
    rows = cursor.execute(query, params).fetchall()
    for row in rows:
        print(row)

# Different SELECT operations.
show_query("1. All students", "SELECT * FROM students")
show_query("2. Students from Pune", "SELECT student_name, city FROM students WHERE city = ?", ("Pune",))
show_query("3. Students ordered by age", "SELECT student_name, age FROM students ORDER BY age DESC")
show_query(
    "4. Join students with courses",
    """
    SELECT s.student_name, c.course_name, c.fee
    FROM enrollments e
    JOIN students s ON e.student_id = s.student_id
    JOIN courses c ON e.course_id = c.course_id
    ORDER BY s.student_name
    """,
)
show_query(
    "5. Course-wise enrollment count",
    """
    SELECT c.course_name, COUNT(*) AS total_students
    FROM enrollments e
    JOIN courses c ON e.course_id = c.course_id
    GROUP BY c.course_name
    ORDER BY total_students DESC
    """,
)

# Update some data.
cursor.execute("UPDATE students SET city = ? WHERE student_name = ?", ("Bengaluru", "Meera"))
cursor.execute("UPDATE courses SET fee = fee + 500 WHERE course_name = ?", ("Python",))
conn.commit()

show_query("6. After update - students", "SELECT * FROM students")
show_query("7. After update - courses", "SELECT * FROM courses")

# Delete some data.
cursor.execute("DELETE FROM enrollments WHERE student_id = ? AND course_id = ?", (2, 101))
cursor.execute("DELETE FROM students WHERE student_name = ?", ("Kabir",))
conn.commit()

show_query("8. After delete - students", "SELECT * FROM students")
show_query("9. After delete - enrollments", "SELECT * FROM enrollments")

conn.close()
print(f"\nDatabase saved to: {db_path.resolve()}")



1. All students
---------------
(1, 'Aarav', 20, 'Pune')
(2, 'Meera', 21, 'Mumbai')
(3, 'Kabir', 22, 'Delhi')
(4, 'Anaya', 20, 'Pune')

2. Students from Pune
---------------------
('Aarav', 'Pune')
('Anaya', 'Pune')

3. Students ordered by age
--------------------------
('Kabir', 22)
('Meera', 21)
('Aarav', 20)
('Anaya', 20)

4. Join students with courses
-----------------------------
('Aarav', 'Python', 5000)
('Anaya', 'Python', 5000)
('Kabir', 'Data Science', 8000)
('Meera', 'SQL', 3500)
('Meera', 'Python', 5000)

5. Course-wise enrollment count
-------------------------------
('Python', 3)
('SQL', 1)
('Data Science', 1)

6. After update - students
--------------------------
(1, 'Aarav', 20, 'Pune')
(2, 'Meera', 21, 'Bengaluru')
(3, 'Kabir', 22, 'Delhi')
(4, 'Anaya', 20, 'Pune')

7. After update - courses
-------------------------
(101, 'Python', 5500)
(102, 'SQL', 3500)
(103, 'Data Science', 8000)

8. After delete - students
--------------------------
(1, 'Aarav', 20, 'Pune')
(2, '